In [2]:
import os
import sys
import re

# =========================================================
# 1. ÉP VERSION PYSPARK (PHẢI CHẠY ĐẦU TIÊN TRƯỚC KHI IMPORT)
# =========================================================
modules_to_remove = [mod for mod in sys.modules if mod.startswith('pyspark') or mod.startswith('py4j')]
for mod in modules_to_remove: 
    del sys.modules[mod]

sys.path = [p for p in sys.path if "/usr/local/spark" not in p]
if "PYTHONPATH" in os.environ: 
    del os.environ["PYTHONPATH"]
    
conda_site_packages = "/opt/conda/lib/python3.13/site-packages"
if conda_site_packages not in sys.path: 
    sys.path.insert(0, conda_site_packages)
    
os.environ["SPARK_HOME"] = os.path.join(conda_site_packages, "pyspark")
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"


# =========================================================
# 2. BÂY GIỜ MỚI IMPORT PYSPARK VÀ KHỞI TẠO SESSION
# =========================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, udf
from pyspark.sql.types import FloatType

spark = SparkSession.builder \
    .appName("compare_score_Pipeline") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://raw-financial-data/iceberg_warehouse") \
    .getOrCreate()


# =========================================================
# 3. VÁ LỖI THỜI GIAN HADOOP
# =========================================================
hadoop_conf = spark._jsc.hadoopConfiguration()
iterator = hadoop_conf.iterator()
while iterator.hasNext():
    entry = iterator.next()
    val = str(entry.getValue()).strip().lower()
    match = re.fullmatch(r"(\d+)([smhd])", val)
    if match:
        num, unit = int(match.group(1)), match.group(2)
        ms_val = num * 1000 if unit == 's' else num * 60000 if unit == 'm' else num * 3600000 if unit == 'h' else num * 86400000
        hadoop_conf.set(entry.getKey(), str(ms_val))

print("✅ Khởi tạo Spark và môi trường hoàn tất!")

✅ Khởi tạo Spark và môi trường hoàn tất!


In [3]:
# =========================================================
# 4. ĐỌC DỮ LIỆU TỪ ICEBERG
# =========================================================
print("📥 Đang đọc dữ liệu từ Iceberg...")

# TODO: ĐỔI TÊN BẢNG CHO PHÙ HỢP VỚI CỦA BẠN
# Giả sử bảng textblob và vader có chung một cột 'id' hoặc 'title_text' để join
table_textblob_name = "my_catalog.processed_zone_finnhub.news_textBlob_tokens_sentiment_score"
table_vader_name = "my_catalog.processed_zone_finnhub.news_VADER_tokens_sentiment_score"

df_textblob = spark.table(table_textblob_name)
df_vader = spark.table(table_vader_name)

📥 Đang đọc dữ liệu từ Iceberg...


In [17]:
# Bổ sung dòng import này vào đầu Cell
from pyspark.sql.functions import col, when
# Gọi toàn bộ bộ hàm của PySpark và đặt tên viết tắt là F
import pyspark.sql.functions as F

# =========================================================
# 5. LÀM SẠCH, GỘP BẢNG VÀ XỬ LÝ SO SÁNH
# =========================================================
print("🔄 Đang làm sạch, gộp bảng và tạo cột so sánh...")

JOIN_KEY = "title_text"

# 1. Lọc trùng lặp
df_textblob_clean = df_textblob.dropDuplicates([JOIN_KEY])
df_vader_clean = df_vader.dropDuplicates([JOIN_KEY])

# 2. Gộp bảng
df_merged = df_textblob_clean.join(df_vader_clean, on=JOIN_KEY, how="inner")

# 3. Tạo các cột so sánh bằng cách gọi F.abs, F.round, F.col...
df_compared = df_merged.withColumn(
    "summary_score_diff",
    F.round(F.abs(F.col("summary_vader_score") - F.col("summary_polarity")), 4)
).withColumn(
    "title_score_diff",
    F.round(F.abs(F.col("title_vader_score") - F.col("title_polarity")), 4)
).withColumn(
    "is_summary_conflict",
    F.when(F.col("summary_polarity_sentiment") != F.col("summary_vader_sentiment"), True).otherwise(False)
).withColumn(
    "is_title_conflict",
    F.when(F.col("title_polarity_sentiment") != F.col("title_vader_sentiment"), True).otherwise(False)
).withColumn(
    "final_summary_sentiment",
    F.when(F.col("is_summary_conflict") == True, F.col("summary_vader_sentiment"))
    .otherwise(F.col("summary_polarity_sentiment"))
)

# =========================================================
# 4. HIỂN THỊ VÀ LƯU DỮ LIỆU CHI TIẾT RA FILE
# =========================================================
print("📊 Đang xử lý hiển thị và lưu dữ liệu chi tiết...")

# 1. Lọc ra các cột cần thiết để lưu (thêm nhãn sentiment để dễ đối chiếu)
df_detail_to_save = df_compared.select(
    JOIN_KEY, 
    "summary_polarity", 
    "summary_polarity_sentiment",  # Nhãn của TextBlob
    "summary_vader_score", 
    "summary_vader_sentiment",     # Nhãn của VADER
    "summary_score_diff",          # Số liệu chênh lệch điểm số
    "is_summary_conflict"          # Cờ đánh dấu có bị ngược nhãn không
)

# 2. Vẫn giữ lệnh in 50 dòng ra màn hình để bạn xem nhanh kết quả
df_detail_to_save.show(50, truncate=False)


# =========================================================
# LỰA CHỌN A: LƯU THÀNH FILE CSV CỤC BỘ TRÊN JUPYTER (Khuyên dùng nếu < 1 triệu dòng)
# =========================================================
# Lệnh này chuyển dữ liệu sang Pandas và ghi thành file CSV ngay trong thư mục chạy code của bạn
file_name_local = "chi_tiet_so_sanh_sentiment.csv"
df_detail_to_save.toPandas().to_csv(file_name_local, index=False)

print(f"✅ Đã xuất và lưu toàn bộ dữ liệu chi tiết vào file: '{file_name_local}'")


# =========================================================
# LỰA CHỌN B: LƯU THÀNH MỘT BẢNG ICEBERG MỚI (Nếu dữ liệu cực kỳ lớn)
# =========================================================
# Nếu tập dữ liệu của bạn quá lớn, không nên chuyển sang Pandas mà hãy lưu thẳng vào Iceberg
"""
detail_table_iceberg = "my_catalog.default.sentiment_detail_comparison"
df_detail_to_save.write \
    .format("iceberg") \
    .mode("overwrite") \
    .save(detail_table_iceberg)

print(f"✅ Đã lưu dữ liệu chi tiết thành bảng Iceberg tại: {detail_table_iceberg}")
"""

🔄 Đang làm sạch, gộp bảng và tạo cột so sánh...
📊 Đang xử lý hiển thị và lưu dữ liệu chi tiết...
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+--------------------------+-------------------+-----------------------+------------------+-------------------+
|title_text                                                                                                                                                            |summary_polarity|summary_polarity_sentiment|summary_vader_score|summary_vader_sentiment|summary_score_diff|is_summary_conflict|
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+--------------------------+-------------------+-----------------------+------------------+-------------------+
|aalo atomics s

'\ndetail_table_iceberg = "my_catalog.default.sentiment_detail_comparison"\ndf_detail_to_save.write     .format("iceberg")     .mode("overwrite")     .save(detail_table_iceberg)\n\nprint(f"✅ Đã lưu dữ liệu chi tiết thành bảng Iceberg tại: {detail_table_iceberg}")\n'

In [14]:
import pyspark.sql.functions as F

# =========================================================
# 6. TÍNH TOÁN CÁC CHỈ SỐ SAI SỐ VÀ ĐỒNG THUẬN TRUNG BÌNH
# =========================================================
print("📊 Đang tính toán các chỉ số thống kê so sánh...")

# 1. Tính toán các chỉ số tỷ lệ và sai số trung bình
metrics_df = df_compared.select(
    # A. Sai số tuyệt đối trung bình (MAD) giữa 2 thang điểm
    F.round(F.mean(F.abs(F.col("summary_vader_score") - F.col("summary_polarity"))), 4).alias("Sai_So_Tuyet_Doi_Trung_Binh_MAD"),
    
    # B. Tỷ lệ % đồng thuận nhãn giữa 2 mô hình
    F.round(
        F.mean(F.when(F.col("is_summary_conflict") == False, 1.0).otherwise(0.0)) * 100, 2
    ).alias("Ty_Le_Dong_Thuan_Nhan_Phan_Tram"),
    
    # C. Tỷ lệ % xung đột hoàn toàn (Một bên Tích cực, một bên Tiêu cực)
    F.round(
        F.mean(
            F.when(
                ((F.col("summary_polarity_sentiment").contains("Tích cực")) & (F.col("summary_vader_sentiment").contains("Tiêu cực"))) |
                ((F.col("summary_polarity_sentiment").contains("Tiêu cực")) & (F.col("summary_vader_sentiment").contains("Tích cực"))), 
                1.0
            ).otherwise(0.0)
        ) * 100, 2
    ).alias("Ty_Le_Xung_Dot_Trai_Nguoc_Phan_Tram")
)

# Hiển thị bảng số liệu tổng hợp
print("\n📋 BẢNG SO SÁNH SAI SỐ VÀ ĐỘ CHÍNH XÁC TƯƠNG ĐỐI:")
metrics_df.show(truncate=False)

# 2. Tính Hệ số tương quan Pearson giữa 2 chuỗi điểm số liên tục
# Giá trị chạy từ -1 đến 1. Càng gần 1 chứng tỏ xu hướng tăng giảm điểm rất đồng điệu.
correlation_value = df_compared.stat.corr("summary_vader_score", "summary_polarity")
print(f"📈 Hệ số tương quan tuyến tính Pearson giữa VADER và TextBlob: {correlation_value:.4f}")

📊 Đang tính toán các chỉ số thống kê so sánh...

📋 BẢNG SO SÁNH SAI SỐ VÀ ĐỘ CHÍNH XÁC TƯƠNG ĐỐI:
+-------------------------------+-------------------------------+-----------------------------------+
|Sai_So_Tuyet_Doi_Trung_Binh_MAD|Ty_Le_Dong_Thuan_Nhan_Phan_Tram|Ty_Le_Xung_Dot_Trai_Nguoc_Phan_Tram|
+-------------------------------+-------------------------------+-----------------------------------+
|0.35                           |52.34                          |6.42                               |
+-------------------------------+-------------------------------+-----------------------------------+

📈 Hệ số tương quan tuyến tính Pearson giữa VADER và TextBlob: 0.3649


In [16]:
# ---------------------------------------------------------
# 7. GỘP CHỈ SỐ VÀ XUẤT FILE BÁO CÁO (CSV)
# ---------------------------------------------------------
print("💾 Đang xuất báo cáo tổng hợp ra file...")

# 1. Thêm hệ số tương quan vào bảng metrics để gom chung thành 1 dòng duy nhất
final_report_df = metrics_df.withColumn(
    "He_So_Tuong_Quan_Pearson", 
    F.lit(round(correlation_value, 4))
)

# Hiển thị lại bảng hoàn chỉnh trước khi lưu
final_report_df.show(truncate=False)

# ==========================================
# CÁCH 1: LƯU THÀNH FILE CỤC BỘ TRÊN JUPYTER (Dễ nhất)
# ==========================================
# Cách này chuyển 1 dòng dữ liệu sang Pandas và lưu thành file CSV ngay trong thư mục code của bạn
final_report_df.toPandas().to_csv("bao_cao_so_sanh_sentiment.csv", index=False)
print("✅ Đã lưu thành công file 'bao_cao_so_sanh_sentiment.csv' vào thư mục hiện tại!")


# ==========================================
# CÁCH 2: LƯU THẲNG LÊN MINIO/S3 BẰNG SPARK (Dành cho Data Pipeline)
# ==========================================
# Nếu bạn muốn lưu file này lên Data Lake (MinIO) cùng chỗ với Iceberg
"""
output_s3_path = "s3a://raw-financial-data/reports/bao_cao_so_sanh_sentiment"

final_report_df.coalesce(1).write \
    .format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(output_s3_path)
    
print(f"✅ Đã lưu báo cáo lên MinIO tại: {output_s3_path}")
"""

💾 Đang xuất báo cáo tổng hợp ra file...
+-------------------------------+-------------------------------+-----------------------------------+------------------------+
|Sai_So_Tuyet_Doi_Trung_Binh_MAD|Ty_Le_Dong_Thuan_Nhan_Phan_Tram|Ty_Le_Xung_Dot_Trai_Nguoc_Phan_Tram|He_So_Tuong_Quan_Pearson|
+-------------------------------+-------------------------------+-----------------------------------+------------------------+
|0.35                           |52.34                          |6.42                               |0.3649                  |
+-------------------------------+-------------------------------+-----------------------------------+------------------------+

✅ Đã lưu thành công file 'bao_cao_so_sanh_sentiment.csv' vào thư mục hiện tại!


'\noutput_s3_path = "s3a://raw-financial-data/reports/bao_cao_so_sanh_sentiment"\n\nfinal_report_df.coalesce(1).write     .format("csv")     .option("header", "true")     .mode("overwrite")     .save(output_s3_path)\n\nprint(f"✅ Đã lưu báo cáo lên MinIO tại: {output_s3_path}")\n'